In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Synteza operacji unitarnych

<details>
<summary><b>Wersje pakietów</b></summary>

Kod na tej stronie został opracowany przy użyciu poniższych wymagań.
Zalecamy korzystanie z tych lub nowszych wersji.

```
qiskit[all]~=2.3.0
```
</details>
Operacja unitarna opisuje zachowującą normę zmianę układu kwantowego.
Dla $n$ Qubitów ta zmiana jest opisana przez macierz $2^n \times 2^n$-wymiarową, zespoloną macierz $U$, której sprzężenie hermitowskie równa się odwrotności, to znaczy $U^\dagger U = \mathbb{1}$.

Synteza konkretnych operacji unitarnych w zestaw Gate kwantowych jest podstawowym zadaniem stosowanym, na przykład, w projektowaniu i zastosowaniu algorytmów kwantowych lub podczas kompilacji Circuit kwantowych.

Choć efektywna synteza jest możliwa dla pewnych klas macierzy unitarnych — takich jak te złożone z Gate Clifforda lub mające strukturę iloczynu tensorowego — większość macierzy unitarnych nie należy do tych kategorii.
Dla ogólnych macierzy unitarnych synteza jest złożonym zadaniem, którego koszty obliczeniowe rosną wykładniczo wraz z liczbą Qubitów.
Dlatego, jeśli znasz efektywną dekompozycję macierzy unitarnej, którą chcesz zaimplementować, prawdopodobnie będzie ona lepsza niż synteza ogólna.

> **Note:** Jeśli żadna dekompozycja nie jest dostępna, Qiskit SDK dostarcza narzędzi do jej znalezienia.
>     Należy jednak pamiętać, że generalnie generuje to głębokie Circuit, które mogą być nieodpowiednie do uruchamiania na zaszumionych komputerach kwantowych.

In [1]:
import numpy as np
from qiskit import QuantumCircuit

U = 0.5 * np.array(
    [[1, 1, 1, 1], [-1, 1, -1, 1], [-1, -1, 1, 1], [-1, 1, 1, -1]]
)

circuit = QuantumCircuit(2)
circuit.unitary(U, circuit.qubits)

## Re-synthesis for circuit optimization

Sometimes it is beneficial to re-synthesize a long series of single- and two-qubit gates, if the length can be reduced. For example, the following circuit uses three two-qubit gates.

In [2]:
from qiskit import QuantumRegister, ClassicalRegister, QuantumCircuit

qreg_q = QuantumRegister(2, "q")
creg_c = ClassicalRegister(4, "c")
circuit = QuantumCircuit(qreg_q, creg_c)

circuit.h(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.sx(qreg_q[1])
circuit.cz(qreg_q[0], qreg_q[1])
circuit.x(qreg_q[1])
circuit.x(qreg_q[0])
circuit.cx(qreg_q[0], qreg_q[1])
circuit.h(qreg_q[0])
circuit.draw("mpl")

<Image src="../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg" alt="Output of the previous code cell" />

## Ponowna synteza w celu optymalizacji Circuit
Czasem korzystne jest ponowne zsyntetyzowanie długiej serii jednoQubitowych i dwuQubitowych Gate, jeśli można zmniejszyć ich liczbę. Na przykład poniższy Circuit używa trzech dwuQubitowych Gate.

In [3]:
from qiskit.quantum_info import Operator

# compute unitary matrix of circuit
U = Operator(circuit)

# re-synthesize
better_circuit = QuantumCircuit(2)
better_circuit.unitary(U, range(2))
better_circuit.decompose().draw()

global phase: 6.2071
      ┌───────────────┐         ┌────────────────┐ 
q_0: ─┤ U(π/2,π/2,-π) ├────■────┤ U(π/2,-π,-π/2) ├─
     ┌┴───────────────┴─┐┌─┴─┐┌─┴────────────────┴┐
q_1: ┤ U(1.7229,π/2,-π) ├┤ X ├┤ U(π/2,0.15207,-π) ├
     └──────────────────┘└───┘└───────────────────┘

![Output of the previous code cell](../docs/images/guides/synthesize-unitary-operators/extracted-outputs/85b63631-b958-48ac-9951-4fa65222a6e1-0.svg)

Jednak po ponownej syntezie z użyciem poniższego kodu potrzebuje jedynie pojedynczej Gate CX. (Tutaj używamy metody `QuantumCircuit.decompose()`, aby lepiej wizualizować Gate użyte do ponownej syntezy macierzy unitarnej.)